In [1]:
import pandas as pd
import os
import requests
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

In [ ]:
SAVE_DIR = "Images"
os.makedirs(SAVE_DIR, exist_ok=True)
airbnb = pd.read_csv('Airbnb_NYC.csv')

def is_valid_image(path):
    """
    Check if an image file is valid (not corrupted).
    """
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except:
        return False


def download_image(url, idx):

    file_path = os.path.join(SAVE_DIR, f"img_{idx}.jpg")

    # Skip if already downloaded and valid
    if os.path.exists(file_path):
        if is_valid_image(file_path):
            return True
        else:
            os.remove(file_path)

    try:
        response = requests.get(url, timeout=3.5)
        response.raise_for_status()

        img = Image.open(BytesIO(response.content)).convert("RGB")

        img.save(file_path, "JPEG", quality=90)

        # Verify saved image
        if not is_valid_image(file_path):
            os.remove(file_path)
            return False

        return True

    except Exception:
        return False


def download_images_parallel(urls, max_workers=12):

    results = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:

        futures = {
            executor.submit(download_image, url, idx): idx
            for idx, url in enumerate(urls)
        }

        with tqdm(total=len(urls), desc="Downloading images") as pbar:

            for future in as_completed(futures):
                result = future.result()
                results.append(result)
                pbar.update(1)

    return results


# Get URLs from dataframe
urls = airbnb["picture_url"].tolist()

# Download images
results = download_images_parallel(urls)

# Summary
total = len(results)
success = sum(results)
failed = total - success

print(f"\nTotal images: {total}")
print(f"Downloaded successfully: {success}")
print(f"Failed downloads: {failed}")


# Add image paths to dataframe
airbnb["image_path"] = [
    os.path.join(SAVE_DIR, f"img_{i}.jpg") if results[i] else None
    for i in range(len(results))
]

# Remove rows where image failed
airbnb_clean = airbnb.dropna(subset=["image_path"]).copy()

print(f"\nOriginal rows: {len(airbnb)}")
print(f"Clean rows with images: {len(airbnb_clean)}")